In [22]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
     

import torchvision
import torchvision.transforms as transforms

from models import *
use_gpu = torch.cuda.is_available()
batch_size = 128
model_name = "vgg16_quant_iter1"
model = VGG16_quant()

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

In [9]:
import importlib
import models.vgg_quant as vgg_quant

importlib.reload(vgg_quant)

<module 'models.vgg_quant' from 'C:\\Users\\derri\\ECE284_Project_Team7\\part1_vgg16\\models\\vgg_quant.py'>

In [23]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))

trainloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))


testloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [24]:
print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input = input.cuda(non_blocking=True)
        target = target.cuda(non_blocking=True)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input = input.cuda(non_blocking=True)
            target = target.cuda(non_blocking=True)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))

In [14]:
def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [38, 55]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.02      

In [15]:
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))

In [26]:
lr = 4e-2
weight_decay = 1e-4
epochs = 10
best_prec = 0
cudnn.benchmark = True
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True
fdir = 'result_checkpoints/'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)
#restarting
for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 0.282 (0.282)	Data 0.078 (0.078)	Loss 1.9439 (1.9439)	Prec 24.219% (24.219%)
Epoch: [0][100/391]	Time 0.026 (0.034)	Data 0.000 (0.001)	Loss 1.7657 (1.8657)	Prec 28.906% (27.785%)
Epoch: [0][200/391]	Time 0.024 (0.031)	Data 0.000 (0.001)	Loss 1.7988 (1.8564)	Prec 29.688% (28.339%)
Epoch: [0][300/391]	Time 0.029 (0.031)	Data 0.000 (0.001)	Loss 1.7573 (1.8472)	Prec 35.156% (29.044%)
Validation starts
Test: [0/79]	Time 0.060 (0.060)	Loss 1.8572 (1.8572)	Prec 25.781% (25.781%)
 * Prec 32.080% 
best acc: 32.080000
Epoch: [1][0/391]	Time 0.100 (0.100)	Data 0.065 (0.065)	Loss 1.7425 (1.7425)	Prec 32.812% (32.812%)
Epoch: [1][100/391]	Time 0.024 (0.030)	Data 0.000 (0.001)	Loss 1.8008 (1.7760)	Prec 29.688% (32.712%)
Epoch: [1][200/391]	Time 0.027 (0.030)	Data 0.000 (0.001)	Loss 1.6754 (1.7694)	Prec 33.594% (33.065%)
Epoch: [1][300/391]	Time 0.041 (0.030)	Data 0.000 (0.001)	Loss 1.8943 (1.7557)	Prec 28.125% (33.591%)
Validation starts
Test: [0/79]	Time 0.053 (0.053)	Loss 1.

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def verify_psum_recovered(model, testloader, num_images=1):

    # --------------------------------------------------
    # 1. Identify the squeezed layer and following layer
    # --------------------------------------------------
    conv = model.features[27]
    next_conv = model.features[29]

    assert conv.in_channels == 8
    assert conv.out_channels == 8
    assert conv.bit == 4
    assert conv.bias is None

    assert isinstance(model.features[28], nn.ReLU)
    assert isinstance(next_conv, nn.Conv2d)
    assert next_conv.in_channels == 8

    # --------------------------------------------------
    # 2. Register forward prehooks
    # --------------------------------------------------
    stored = {}

    def save_conv_input(module, inputs):
        stored["conv_input"] = inputs[0].detach().clone()

    def save_next_input(module, inputs):
        stored["prehook_next"] = inputs[0].detach().clone()

    hook1 = conv.register_forward_pre_hook(save_conv_input)
    hook2 = next_conv.register_forward_pre_hook(save_next_input)

    was_training = model.training
    original_tf32 = torch.backends.cudnn.allow_tf32

    try:
        model.eval()

        # Disable TF32 for a more accurate FP32 reference
        torch.backends.cudnn.allow_tf32 = False

        # --------------------------------------------------
        # 3. Execute forward pass on test images
        # --------------------------------------------------
        images, _ = next(iter(testloader))

        device = next(model.parameters()).device
        images = images[:num_images].to(device)

        with torch.no_grad():
            _ = model(images)

            # --------------------------------------------------
            # 4. Obtain actual quantized activations and weights
            # --------------------------------------------------

            # Quantize prehooked input using the layer's quantizer
            x_q = conv.act_alq(
                stored["conv_input"],
                conv.act_alpha
            ).detach()

            # Quantized weights saved during the forward pass
            w_q = conv.weight_q.detach().clone()

            # --------------------------------------------------
            # 5. Calculate quantization step sizes
            # --------------------------------------------------

            # 4-bit unsigned activations: 0 to 15
            delta_x = (
                conv.act_alpha.detach().double()
                / (2**conv.bit - 1)
            )

            # 4-bit signed weights: -7 to 7
            delta_w = (
                conv.weight_quant.wgt_alpha.detach().double()
                / (2**(conv.bit - 1) - 1)
            )

            # --------------------------------------------------
            # 6. Recover integer activations and weights
            # --------------------------------------------------

            x_int = torch.round(
                x_q.double() / delta_x
            ).to(torch.int64)

            w_int = torch.round(
                w_q.double() / delta_w
            ).to(torch.int64)

            # Verify integer ranges
            assert x_int.min().item() >= 0
            assert x_int.max().item() <= 15

            assert w_int.min().item() >= -7
            assert w_int.max().item() <= 7

            # --------------------------------------------------
            # 7. Perform integer-valued convolution
            # --------------------------------------------------

            # FP64 represents these small integers exactly.
            # Integer products and sums are therefore exact
            # for this convolution's accumulation range.

            psum_int = F.conv2d(
                x_int.double(),
                w_int.double(),
                bias=None,
                stride=conv.stride,
                padding=conv.padding,
                dilation=conv.dilation,
                groups=conv.groups
            )

            # --------------------------------------------------
            # 8. Recover floating-point psum and apply ReLU
            # --------------------------------------------------

            psum_recovered = F.relu(
                psum_int * delta_x * delta_w
            )

            # --------------------------------------------------
            # 9. Get the next layer's prehooked input
            # --------------------------------------------------

            prehook_next = stored["prehook_next"].double()

            assert psum_recovered.shape == prehook_next.shape

            # --------------------------------------------------
            # 10. Compare recovered psum with hooked reference
            # --------------------------------------------------

            abs_error = (
                psum_recovered - prehook_next
            ).abs()

            max_error = abs_error.max().item()
            mean_error = abs_error.mean().item()

            # --------------------------------------------------
            # 11. Display verification results
            # --------------------------------------------------

            print("=== Layer Information ===")
            print("Squeezed layer:", conv)
            print("Input shape:", tuple(x_int.shape))
            print("Weight shape:", tuple(w_int.shape))
            print("Output shape:", tuple(psum_recovered.shape))

            print("\n=== Quantization Parameters ===")
            print("delta_x:", delta_x.item())
            print("delta_w:", delta_w.item())

            print("\n=== Integer Ranges ===")
            print("x_int min/max:",
                  x_int.min().item(),
                  x_int.max().item())

            print("w_int min/max:",
                  w_int.min().item(),
                  w_int.max().item())

            print("\n=== Verification Results ===")
            print("Maximum absolute error:", max_error)
            print("Mean absolute error:", mean_error)

            print("Ideal threshold (< 1e-5):",
                  max_error < 1e-5)

            print("Project threshold (< 1e-3):",
                  max_error < 1e-3)

            print("\n=== Sample Outputs ===")
            print("psum_recovered:",
                  psum_recovered.flatten()[:8].tolist())

            print("prehook_next:",
                  prehook_next.flatten()[:8].tolist())

            if max_error < 1e-3:
                print("\nPASS: psum_recovered matches prehook")
            else:
                print("\nFAIL: Error exceeds project threshold")

            return {
                "x_int": x_int,
                "w_int": w_int,
                "psum_int": psum_int,
                "psum_recovered": psum_recovered,
                "prehook_next": prehook_next,
                "max_error": max_error,
                "mean_error": mean_error,
                "delta_x": delta_x,
                "delta_w": delta_w
            }

    finally:
        # Remove hooks and restore original model settings
        hook1.remove()
        hook2.remove()

        torch.backends.cudnn.allow_tf32 = original_tf32
        model.train(was_training)


# Run verification on the existing trained model
results = verify_psum_recovered(
    model,
    testloader,
    num_images=4
)

=== Layer Information ===
Squeezed layer: QuantConv2d(
  8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)
Input shape: (4, 8, 4, 4)
Weight shape: (8, 8, 3, 3)
Output shape: (4, 8, 4, 4)

=== Quantization Parameters ===
delta_x: 0.33859049479166664
delta_w: 0.37987927028111046

=== Integer Ranges ===
x_int min/max: 0 15
w_int min/max: -7 3

=== Verification Results ===
Maximum absolute error: 3.1119345571539725e-06
Mean absolute error: 2.403363046329975e-07
Ideal threshold (< 1e-5): True
Project threshold (< 1e-3): True

=== Sample Outputs ===
psum_recovered: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0869642420538828, 2.5724702017115693]
prehook_next: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0869648456573486, 2.5724709033966064]

PASS: psum_recovered matches prehook
